## Minimal example of calculating NPQ-corrected fluorescence from float data

This notebook defines the function for calculating NPQ-corrected fluorescence from uncalibrated float `CHLA_FLUORESCENCE`, and applies it to some data from float ID 2903884 in the equatorial Pacific.

### Import packages, define function, read in float data

In [ ]:
import xarray as xr
import numpy as np

import matplotlib.pyplot as plt

In [ ]:
from scripts.npqc import calculate_NPQ_fluo
from scripts.par15 import par15
from scripts.mld_calculator import mld_calc

In [ ]:
ds = xr.open_dataset('data/2903884_Sprof.nc')

### Calculate and plot for the first 10 profiles

In [ ]:
ds = par15(ds).rename({'par_15_pressure': 'par_depth'})
ds = mld_calc(ds)

In [ ]:
ds = ds#.isel(N_PROF=slice(30))
ds = ds.dropna(dim='N_PROF', subset=['par_depth', 'ml_depth'])
ds = calculate_NPQ_fluo(ds)

In [ ]:
ds

In [ ]:
fig, ax = plt.subplots()
plt.plot(ds['fluo_smooth'].T, ds['PRES_ADJUSTED'].T, '-')
plt.plot(ds['fluo_npqc'].T, ds['PRES_ADJUSTED'].T, '--')
# match line colors
h1 = ax.get_children()
for hh in range(13, 24):
    h1[hh].set_color(h1[hh-12].get_color())
ax.set_ylim([250, 0])

### Calculate NPQ
Eq. 2 from [Schallenberg et al. 2022](https://doi.org/10.1029/2021GL097616)

In [ ]:
ds['NPQ'] = (ds['fluo_npqc']-ds['fluo_smooth'])/ds['fluo_smooth']

In [ ]:
fig, ax = plt.subplots()
ds['par_depth'].plot.line('-x', label='PAR depth')
ds['ml_depth'].plot.line('-x', label='MLD')
ds['NPQ_depth'].plot.line('-x', label='NPQ depth')
ax.set_ylabel('Depth [dbar]')
plt.legend()
ax.set_ylim(ax.get_ylim()[::-1]);